In [1]:
import requests
import pandas as pd

# 1. Definindo o endereço da API (Endpoint)
# Vamos buscar o histórico de preços do Bitcoin (BTC) em Dólares (USD) dos últimos 30 dias
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days=30"

# 2. Fazendo a requisição GET (O 'Extract')
response = requests.get(url)

# 3. Verificando se a comunicação foi bem-sucedida (Status 200 = Sucesso)
if response.status_code == 200:
    dados_brutos = response.json()
    print("✅ Extração realizada com sucesso!\n")
    
    # A API retorna um dicionário JSON com 3 listas: 'prices', 'market_caps' e 'total_volumes'
    # Vamos dar uma espiada apenas nos 3 primeiros registros da lista de preços:
    print("Espiando o formato dos dados de preço:")
    print(dados_brutos['prices'][:3])
    
else:
    print(f"❌ Falha na extração. Código de Erro: {response.status_code}")

✅ Extração realizada com sucesso!

Espiando o formato dos dados de preço:
[[1774576918450, 68987.91604743515], [1774580550629, 68861.18343940613], [1774584105099, 68732.87385203048]]


In [2]:
# 1. Transformando a lista JSON em uma tabela (DataFrame)
df = pd.DataFrame(dados_brutos['prices'], columns=['timestamp', 'preco_usd'])

# 2. Convertendo aquele número gigante (timestamp) para uma data normal
# O parâmetro unit='ms' avisa o Pandas que o número está em milissegundos
df['data'] = pd.to_datetime(df['timestamp'], unit='ms')

# 3. Arredondando o preço para ficar com cara de dinheiro (2 casas decimais)
df['preco_usd'] = df['preco_usd'].round(2)

# 4. Limpando a casa: vamos jogar fora a coluna timestamp velha e organizar a ordem
df = df[['data', 'preco_usd']]

# 5. O comando display() mostra a tabela formatada bonitinha no Jupyter
display(df.head())

,data,preco_usd
0,2026-03-27 02:01:58.450,68987.92
1,2026-03-27 03:02:30.629,68861.18
2,2026-03-27 04:01:45.099,68732.87
3,2026-03-27 05:00:10.314,68544.83
4,2026-03-27 06:02:22.784,68584.58


In [5]:
# 1. Definindo o caminho onde o arquivo será salvo
# Como o nosso notebook está dentro da pasta 'notebooks', usamos '../' para voltar para a raiz e entrar na pasta 'data'
caminho_arquivo = '../data/historico_btc.csv'

# 2. Exportando o DataFrame para um arquivo CSV (O 'Load')
# index=False evita que o Pandas crie uma coluna extra com a numeração das linhas
df.to_csv(caminho_arquivo, index=False, sep=';')

print(f"💾 Carga finalizada! Arquivo salvo em: {caminho_arquivo}")

💾 Carga finalizada! Arquivo salvo em: ../data/historico_btc.csv


In [7]:
import requests
import pandas as pd
import time

# List of assets to be monitored
assets = ['bitcoin', 'ethereum', 'solana', 'cardano']
data_frames = []

print("Starting multi-asset ETL pipeline...")

for asset in assets:
    # API URL for market chart data (last 30 days)
    url = f"https://api.coingecko.com/api/v3/coins/{asset}/market_chart?vs_currency=usd&days=30"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        raw_data = response.json()
        
        # Transformation: Loading raw price list into a DataFrame
        df_temp = pd.DataFrame(raw_data['prices'], columns=['timestamp', 'price_usd'])
        
        # Adding metadata for identification
        df_temp['asset'] = asset
        df_temp['date'] = pd.to_datetime(df_temp['timestamp'], unit='ms')
        df_temp['price_usd'] = df_temp['price_usd'].round(2)
        
        # Selecting final schema
        data_frames.append(df_temp[['date', 'asset', 'price_usd']])
        
        # Sleep to avoid API rate limiting (HTTP 429)
        time.sleep(1)
    else:
        print(f"Failed to fetch {asset}. Status Code: {response.status_code}")

# Concatenating all individual DataFrames into one master table
final_df = pd.concat(data_frames, ignore_index=True)

print("ETL Process finished successfully.")
display(final_df.sample(5))

Starting multi-asset ETL pipeline...
ETL Process finished successfully.


,date,asset,price_usd
2452,2026-04-07 12:02:53.817,cardano,0.24
1963,2026-04-17 04:00:23.647,solana,87.73
1815,2026-04-11 05:34:57.754,solana,84.27
1291,2026-04-19 10:00:35.292,ethereum,2310.42
2888,2026-04-25 10:01:04.660,cardano,0.25


In [11]:
# This ensures a logical timeline for each cryptocurrency
final_df = final_df.sort_values(by=['date', 'asset'], ascending=[True, True])
# Define the output path
output_path = '../data/crypto_history.csv'

# Save the multi-asset DataFrame to CSV
# Using ';' as separator to maintain compatibility with different region settings
try:
    final_df.to_csv(output_path, index=False, sep=';')
    print(f"💾 Load finished! File saved at: {output_path}")
except PermissionError:
    print("❌ Error: Please close the CSV file in Excel before running this cell.")

💾 Load finished! File saved at: ../data/crypto_history.csv


In [12]:
import sqlite3

def save_to_database(df, db_name='../data/finance_data.db'):
    """
    Saves the processed DataFrame into a SQLite database.
    If the table already exists, it appends the new data.
    """
    try:
        # Create a connection to the database
        conn = sqlite3.connect(db_name)
        
        # Load the DataFrame into the 'crypto_prices' table
        # if_exists='append' allows adding new data without deleting old ones
        df.to_sql('crypto_prices', conn, if_exists='append', index=False)
        
        # Close the connection
        conn.close()
        print(f"✅ Data successfully persisted in {db_name}")
    except Exception as e:
        print(f"❌ Database error: {e}")

# Persistence execution
save_to_database(final_df)

✅ Data successfully persisted in ../data/finance_data.db


In [38]:
def query_crypto_data(asset_name=None):
    """
    Executes a SQL query to retrieve data from the SQLite database.
    If an asset_name is provided, it filters the results.
    """
    conn = sqlite3.connect('../data/finance_data.db')
    
    query = "SELECT *  FROM crypto_prices" 
    
    if asset_name:
        # Using f-string for the query (Safe for local lab projects)
        query += f" WHERE asset = '{asset_name}'"
    
    # Read the SQL query directly into a pandas DataFrame
    df_result = pd.read_sql_query(query, conn)
    conn.close()
    
    return df_result

# Example: Get only Ethereum data
print(f"Total de registros encontrados: {eth_data.shape[0]}")
eth_data = query_crypto_data('bitcoin')
display(eth_data.head(10))


Total de registros encontrados: 727


,date,asset,price_usd
0,2026-03-27 03:02:30.629000,bitcoin,68861.18
1,2026-03-27 04:01:45.099000,bitcoin,68732.87
2,2026-03-27 05:00:10.314000,bitcoin,68544.83
3,2026-03-27 06:02:22.784000,bitcoin,68584.58
4,2026-03-27 07:02:32.479000,bitcoin,68614.74
5,2026-03-27 08:02:21.043000,bitcoin,68433.30
6,2026-03-27 09:02:08.660000,bitcoin,67896.72
7,2026-03-27 10:02:05.619000,bitcoin,67666.10
8,2026-03-27 11:02:32.756000,bitcoin,66462.11
9,2026-03-27 12:01:18.669000,bitcoin,66666.37
